***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [8. 校准](8_0_introduction.ipynb)
    * 上一节： [8.3 第二代校准（2GC）](8_3_2gc.ipynb)
    * 下一节： [8.5 延伸阅读与参考文献](8_5_further_reading_and_references.ipynb)

***


导入标准模块:


In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import HTML, display
except ImportError:
    HTML = None
    display = None

STYLE_PATH = Path("../style/course.css")
TOGGLE_PATH = Path("../style/code_toggle.html")

if HTML is not None and display is not None:
    if STYLE_PATH.exists():
        display(HTML(f"<style>{STYLE_PATH.read_text(encoding='utf-8')}</style>"))
    if TOGGLE_PATH.exists():
        display(HTML(TOGGLE_PATH.read_text(encoding="utf-8")))

plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=3, suppress=True)

RNG = np.random.default_rng(20260419)


当误差明显依赖天空方向时，方向无关自校准就不再足够。主波束变化、指向误差、电离层相位屏、对流层水汽以及宽场极化泄漏，都会让“同一面天线只有一个复增益”这个近似失效。

第三代校准（3GC）的任务，就是在可计算成本可接受的前提下，把这些方向依赖效应纳入模型或求解器，使残差不再随着天空位置系统性增长。


***


## 8.4 第三代校准（3GC）：方向依赖自校准

这一节不追求穷尽所有 3GC 算法，而是建立三个最重要的判断：

- 什么时候 2GC 的方向无关近似已经失效；
- 为什么“一个全场单一增益”无法同时解释多个方向上的偏差；
- 3GC 中常见的几条路线分别解决什么问题。


In [ ]:
def baseline_pairs(nant):
    return [(p, q) for p in range(nant) for q in range(p + 1, nant)]


def baseline_u(ant_x_m, hour_angle_h, wavelength_m=0.214):
    pairs = baseline_pairs(len(ant_x_m))
    hour_angle_rad = np.deg2rad(15.0 * hour_angle_h)
    u = np.zeros((hour_angle_h.size, len(pairs)))
    for ti, ha in enumerate(hour_angle_rad):
        for bi, (p, q) in enumerate(pairs):
            u[ti, bi] = (ant_x_m[q] - ant_x_m[p]) * np.sin(ha) / wavelength_m
    return pairs, u


def direction_visibilities(u, fluxes, ls):
    vis = []
    for flux, ll in zip(fluxes, ls):
        vis.append(flux * np.exp(-2j * np.pi * u * ll))
    return np.stack(vis, axis=-1)


def beam_gain(l, offset, sigma):
    return np.exp(-0.5 * ((l - offset) / sigma) ** 2)


def dirty_image_1d(u, vis, l_grid):
    u_flat = np.concatenate([u.ravel(), -u.ravel()])
    vis_flat = np.concatenate([vis.ravel(), np.conj(vis.ravel())])
    phase = np.exp(2j * np.pi * u_flat[:, None] * l_grid[None, :])
    return np.real(vis_flat @ phase / vis_flat.size)


ant_x = np.array([0.0, 42.0, 110.0, 205.0, 325.0, 470.0])
times_h = np.linspace(-2.8, 2.8, 42)
pairs, u = baseline_u(ant_x, times_h)
nant = ant_x.size

fluxes = np.array([1.0, 0.36, 0.22])
ls = np.array([0.0, 0.018, -0.026])
source_vis = direction_visibilities(u, fluxes, ls)

static_offsets = np.array([0.000, 0.0015, -0.0010, 0.0022, -0.0018, 0.0012])
dynamic_offsets = 0.0012 * np.sin(1.4 * times_h[:, None] + np.linspace(0.0, np.pi, nant)[None, :])
pointing_offsets = static_offsets[None, :] + dynamic_offsets
sigma = 0.020

beam = np.zeros((times_h.size, nant, len(ls)))
for si, ll in enumerate(ls):
    beam[:, :, si] = beam_gain(ll, pointing_offsets, sigma)

data = np.zeros((times_h.size, len(pairs)), dtype=complex)
for bi, (p, q) in enumerate(pairs):
    data[:, bi] = np.sum(beam[:, p, :] * beam[:, q, :] * source_vis[:, bi, :], axis=1)

model_di = np.sum(source_vis, axis=-1)
model_dd = np.zeros_like(model_di)
for bi, (p, q) in enumerate(pairs):
    model_dd[:, bi] = np.sum(beam[:, p, :] * beam[:, q, :] * source_vis[:, bi, :], axis=1)

residual_di = data - model_di
residual_dd = data - model_dd
l_grid = np.linspace(-0.05, 0.05, 700)
image_di = dirty_image_1d(u, residual_di, l_grid)
image_dd = dirty_image_1d(u, residual_dd, l_grid)


### 8.4.1 从主波束与指向误差理解方向依赖效应 <a id='cal:sec:p_versus_h'></a>

下面先看一个最常见的 3GC 来源：主波束加上小的指向偏差。对位于视场中心的源，波束变化往往较弱；但对离轴源，同样大小的指向偏差就会被放大成明显的通量偏差和相位/振幅残差。


In [ ]:
l_axis = np.linspace(-0.05, 0.05, 500)
example_antennas = [0, 2, 4]

fig, axes = plt.subplots(1, 2, figsize=(11, 4.2))

for ant in example_antennas:
    axes[0].plot(l_axis, beam_gain(l_axis, static_offsets[ant], sigma), lw=2.0, label=f"antenna {ant}")
for ll in ls:
    axes[0].axvline(ll, color="black", ls=":", alpha=0.6)
axes[0].set_xlabel("direction cosine l")
axes[0].set_ylabel("primary-beam gain")
axes[0].set_title("Different antennas see different off-axis gains")
axes[0].legend(loc="upper right")

for si, ll in enumerate(ls[1:]):
    axes[1].plot(times_h, beam[:, 3, si + 1], lw=2.0, label=f"source at l={ll:+.3f}")
axes[1].set_xlabel("time [hour]")
axes[1].set_ylabel("beam gain of antenna 3")
axes[1].set_title("Off-axis response also varies with time")
axes[1].legend(loc="upper right")

plt.tight_layout()


一旦不同方向上的增益不再相同，就不可能再用一个全场统一的 $G_p$ 同时解释所有源。这时如果仍强行使用方向无关解，求解器往往会优先照顾最亮的源，而把较弱离轴源附近的残差留在图像里。


### 8.4.2 为什么 2GC 模型无法消除这类残差

下面比较两种模型：

- **DI 模型**：假设全场都共享同一组方向无关增益，不考虑主波束差异；
- **DD 模型**：显式把方向依赖波束项写进每个源的预测可见度。

对于这里构造的数据，DD 模型是“正确模型”，因此 residual 应接近零；而 DI 模型则会在离轴源附近留下结构化残差。


In [ ]:
rms_di = np.sqrt(np.mean(np.abs(residual_di) ** 2))
rms_dd = np.sqrt(np.mean(np.abs(residual_dd) ** 2))

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)

axes[0].plot(l_grid, image_di, color="tab:red", lw=1.7)
for ll in ls:
    axes[0].axvline(ll, color="black", ls=":", alpha=0.6)
axes[0].set_ylabel("residual image")
axes[0].set_title("Residual after a direction-independent model")

axes[1].plot(l_grid, image_dd, color="tab:blue", lw=1.7)
for ll in ls:
    axes[1].axvline(ll, color="black", ls=":", alpha=0.6)
axes[1].set_ylabel("residual image")
axes[1].set_xlabel("direction cosine l")
axes[1].set_title("Residual after a direction-dependent model")

plt.tight_layout()
print(f"Visibility RMS residual: DI model = {rms_di:.4e}, DD model = {rms_dd:.4e}")


这个对比说明的不是“3GC 一定更好”，而是：**当污染本质上是方向依赖时，不升级模型就无法从根本上减少残差。** 此时继续在 2GC 框架里调更复杂的参数，通常只会把问题隐藏，而不会真正解决。


### 8.4.3 3GC 的几条常见路线

在真实软件中，3GC 大致有下面几类思路：

- **物理模型法**：把主波束、电离层屏或指向误差参数化后直接写进 RIME，再联合求解；
- **差分增益 / peeling**：对少数亮离轴源单独建立方向相关增益；
- **faceting / image-domain methods**：把大视场分块，每块近似共享局部响应；
- **A-projection / AW-projection**：在成像卷积核里显式传播主波束和宽场效应；
- **混合策略**：先用物理模型吸收大尺度效应，再对最亮方向做局部修正。

哪条路线更合适，取决于误差类型、亮源分布、计算预算以及阵列本身的方向依赖复杂度。


### 8.4.4 求解器与计算代价 <a id='cal:sec:stef'></a>

进入 3GC 后，计算难点不只在于“未知数更多”，更在于未知数与数据之间的耦合结构更复杂。常见的瓶颈包括：

- 参数数量随方向数、时间和频率网格迅速增长；
- 模型预测需要更频繁地在图像域与可见度域之间往返；
- 若主波束或电离层模型本身带有误差，求解器会出现明显退化。

因此，像 StEFCal 这类高效增益求解器仍然非常重要，但它们通常只是更大 3GC 工作流中的一个模块，而不是完整答案。真正的工程挑战在于：如何把求解器、成像器和物理先验以稳定的方式组合起来。


#### 什么时候应该从 2GC 升级到 3GC？

若你在完成 1GC 和 2GC 后仍持续看到以下症状，就应认真考虑 3GC：

- 残差随视场位置明显变化，而不是全场均匀；
- 图像边缘的动态范围远差于视场中心；
- 离轴亮源周围反复出现条纹、拉伸或负碗状残差；
- 同一源在不同时间、频率或偏振下表现出位置相关的系统误差。

这时继续“微调 2GC”往往收益有限。更合理的做法，是回到误差物理本身，判断是否已经进入了方向依赖校准的适用区间。
